# Day 62.5 - the memory hierarchy and the roofline

Every performance number in this series turned out to be a memory number. This notebook
is the model underneath: the hierarchy measured on this machine, the roofline that
decides whether a kernel is limited by arithmetic or by traffic, and tiling - including
FlashAttention - as the only way to change which side you are on.

In [ ]:
import numpy as np
from memory_hierarchy import (MACHINES, hierarchy_table, hbm, measure_sequential,
                              measure_random, Machine, roofline_table, batch_crossover,
                              matmul_traffic, tile_sweep, flash_io, flash_scaling,
                              little, strided_cost)

for name in MACHINES:
    m = Machine(name)
    print(f"{name} ({MACHINES[name]['vendor']}): {m.flops/1e12:.0f} TFLOP/s, "
          f"{m.bw/1e9:.0f} GB/s, balance {m.balance:.0f} FLOPs/byte")
    for r in hierarchy_table(name):
        print(f"   {r['name']:30} {r['gbs']:9,.0f} GB/s "
              f"({'spec' if r['bandwidth_source'] == 'spec' else 'est.'})"
              f"   {r['ns']:6,.0f} ns (est.)")
print()
from memory_hierarchy import SOURCES
for src in SOURCES:
    print('-', src)

MI300X (AMD, CDNA 3): 1307 TFLOP/s, 5300 GB/s, balance 247 FLOPs/byte
   registers (VGPR per CU)          100,000 GB/s (est.)        1 ns (est.)
   LDS / shared memory per CU        20,000 GB/s (est.)       20 ns (est.)
   L2 per XCD (8 of them)            10,000 GB/s (est.)      200 ns (est.)
   Infinity Cache (MALL)             17,000 GB/s (spec)      300 ns (est.)
   HBM3                               5,300 GB/s (spec)      600 ns (est.)
   host memory over PCIe 5 x16           64 GB/s (spec)    2,000 ns (est.)
   another node over RDMA                50 GB/s (spec)   10,000 ns (est.)
H200 SXM (NVIDIA, Hopper): 990 TFLOP/s, 4800 GB/s, balance 206 FLOPs/byte
   registers (per SM)               100,000 GB/s (est.)        1 ns (est.)
   shared memory per SM              20,000 GB/s (est.)       20 ns (est.)
   L2 cache                           7,000 GB/s (est.)      200 ns (est.)
   HBM3e                              4,800 GB/s (spec)      600 ns (est.)
   host memory over PCIe 5 x16 

## 1. Measured here: in order against at random

Same machine, same bytes. The prefetcher hides the hierarchy from one of them.

In [ ]:
for r in measure_sequential():
    print(f"{r['kb']:7d} KB   {r['gbs']:6.1f} GB/s")
print()
for r in measure_random():
    print(f"{r['kb']:7d} KB   {r['ns_per_access']:6.2f} ns per access")

     64 KB     16.8 GB/s
    256 KB     20.2 GB/s
   1024 KB     24.3 GB/s
   4096 KB     20.8 GB/s
  16384 KB     22.5 GB/s
  65536 KB     19.5 GB/s
 262144 KB     17.9 GB/s

     16 KB     1.65 ns per access
     64 KB     1.17 ns per access
    256 KB     1.13 ns per access
   1024 KB     1.51 ns per access
   4096 KB     7.63 ns per access
  16384 KB    10.81 ns per access
  65536 KB    12.57 ns per access


## 2. The roofline

A machine is two numbers; their ratio is the balance. A kernel below it cannot be
compute-bound however it is written.

In [ ]:
rows, m = roofline_table()
print(f"peak {m.flops/1e12:.0f} TFLOP/s, {m.bw/1e9:.0f} GB/s, balance {m.balance:.0f} FLOPs/byte\n")
for r in rows:
    print(f"{r['name']:36} {r['intensity']:10.2f}  {r['bound']:8} {r['ms']:7.2f} ms")

peak 1307 TFLOP/s, 5300 GB/s, balance 247 FLOPs/byte

decode: one linear (GEMV)                  1.00  memory      0.02 ms
prefill: one linear (GEMM)              3185.78  compute     0.37 ms
decode: attention over KV                  1.00  memory      0.01 ms
prefill: attention (no tiling)           128.00  memory      0.41 ms
RMSNorm                                    1.00  memory      0.01 ms
residual add                               0.17  memory      0.02 ms
softmax over the vocabulary                1.25  memory      0.00 ms
decode: the whole model                    1.00  memory     26.42 ms


## 3. Batching moves a kernel along the roofline

In [ ]:
bc, balance = batch_crossover()
for r in bc:
    print(f"batch {r['batch']:5d}  {r['intensity']:9.2f} FLOPs/byte  {r['bound']:8}"
          f"  {r['per_token_us']:7.2f} us per token")

batch     1       1.00 FLOPs/byte  memory      22.16 us per token
batch     4       4.00 FLOPs/byte  memory       5.54 us per token
batch    16      15.96 FLOPs/byte  memory       1.39 us per token
batch    64      63.43 FLOPs/byte  memory       0.35 us per token
batch   256     247.17 FLOPs/byte  compute      0.09 us per token
batch  1024     896.00 FLOPs/byte  compute      0.09 us per token


## 4. Tiling, and FlashAttention as the same argument

In [ ]:
t = matmul_traffic()
print(f"untiled {t['naive_bytes']/1e9:.1f} GB   tiles of {t['tile']} "
      f"{t['tiled_bytes']/1e9:.2f} GB   {t['ratio']:.0f}x less")
for r in tile_sweep():
    print(f"  T={r['tile']:4d} {'fits' if r['fits'] else 'DOES NOT FIT':>12}  {r['gb']:8.2f} GB")

print()
f = flash_io()
print(f"score matrix {f['score_matrix_gb']:.2f} GB resident; standard moves "
      f"{f['standard_gb']:.2f} GB, tiled {f['tiled_gb']:.2f} GB counting every level "
      f"({f['ratio']:.1f}x) but only {f['tiled_hbm_gb']:.2f} GB reaches HBM "
      f"({f['ratio_hbm']:.1f}x)")
print(f"K+V per head {f['kv_per_head_mb']:.1f} MB vs a {f['cache_mb']:.0f} MB cache: "
      f"fits = {f['kv_fits_cache']}")
print(f"the matrix is {f['memory_ratio']:,.0f}x the shared memory the kernel runs in")
for r in flash_scaling():
    print(f"  seq {r['seq']:6d}  matrix {r['score_matrix_gb']:8.2f} GB  ratio {r['ratio']:.1f}x")

untiled 549.8 GB   tiles of 104 5.35 GB   103x less
  T=   8         fits     68.79 GB
  T=  16         fits     34.43 GB
  T=  32         fits     17.25 GB
  T=  64         fits      8.66 GB
  T= 128 DOES NOT FIT      4.36 GB
  T= 256 DOES NOT FIT      2.21 GB
  T= 512 DOES NOT FIT      1.14 GB

score matrix 1.07 GB resident; standard moves 4.43 GB, tiled 4.36 GB counting every level (1.0x) but only 0.13 GB reaches HBM (33.0x)
K+V per head 2.1 MB vs a 268 MB cache: fits = True
the matrix is 16,384x the shared memory the kernel runs in
  seq    512  matrix     0.02 GB  ratio 1.1x
  seq   2048  matrix     0.27 GB  ratio 1.0x
  seq   8192  matrix     4.29 GB  ratio 1.0x
  seq  32768  matrix    68.72 GB  ratio 1.0x


## 5. Latency and cache lines

In [ ]:
print(little())
for r in strided_cost():
    print(f"stride {r['stride']:3d}  useful {r['useful_mb']:7.2f} MB  fetched {r['fetched_mb']:7.2f} MB"
          f"  {r['efficiency']:6.1%}  {r['seconds']*1e3:5.2f} ms")

{'bandwidth_gbs': 5300.0, 'latency_ns': 600.0, 'bytes_in_flight': 3180000.0, 'requests_in_flight': 24843.75, 'bytes_per_request': 128}
stride   1  useful   16.78 MB  fetched   16.78 MB  100.0%   0.97 ms
stride   2  useful    8.39 MB  fetched   16.78 MB   50.0%   0.85 ms
stride   4  useful    4.19 MB  fetched   16.78 MB   25.0%   0.65 ms
stride   8  useful    2.10 MB  fetched   16.78 MB   12.5%   0.67 ms
stride  16  useful    1.05 MB  fetched   16.78 MB    6.2%   0.72 ms
stride  32  useful    0.52 MB  fetched    8.39 MB    6.2%   0.50 ms


## 6. Tests

In [ ]:
m = Machine()
rows, _ = roofline_table()
by = {r['name']: r for r in rows}
assert by['decode: one linear (GEMV)']['bound'] == 'memory'
assert by['prefill: one linear (GEMM)']['bound'] == 'compute'
bc, _ = batch_crossover()
assert bc[0]['bound'] == 'memory' and bc[-1]['bound'] == 'compute'
t = matmul_traffic(n=1024, tile=64)
assert abs(t['ratio'] - 64) < 3
assert flash_io(seq=16384)['memory_ratio'] > 1000
assert flash_io(seq=16384)['ratio_hbm'] >= flash_io(seq=16384)['ratio']
print('all assertions passed')

all assertions passed
